## 0 · Install dependencies

In [10]:
# !pip install pmdarima lightgbm xgboost -q

## 1 · Imports & Configuration

In [11]:
import os, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_absolute_error, mean_squared_error
import xgboost  as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')

TARGET_COL = "LTE DL User Throughput"
YLABEL     = "LTE DL User Throughput (kbit/s)"
CELL_NAME  = "ZAG_0158_C01_125_l3"
PLOT_DIR   = "throughput_plots"
TABLE_PATH = "comparison_table_Throughput.csv"
HORIZONS   = [6, 24]
os.makedirs(PLOT_DIR, exist_ok=True)
print("Config ready — output dir:", PLOT_DIR)
print("Target KPI  :", TARGET_COL)

Config ready — output dir: throughput_plots
Target KPI  : LTE DL User Throughput


## 2 · Data Loading & Feature Engineering

In [12]:
df = pd.read_csv("../datasets/final_kpi_data.csv")
df = df[['Time', 'Cell Name', 'Radio Site', 'Region',
         'Active Users', 'LTE DL User Throughput', 'DL PRB Load',
         'Cell FDD TDD Indication']]
df['Time'] = pd.to_datetime(df['Time'])
df = df.sort_values(['Cell Name', 'Time'])

# ── Identity encodings ──────────────────────────────────────────────────
# cell_id  : unique per cell (finest granularity)
# site_id  : unique per Radio Site  (groups 2-3 cells sharing the same mast)
# region_id: unique per Region      (groups all sites in the same area)
# Encoding as integer category codes keeps memory low and is compatible
# with tree-based models directly.
df['cell_id']   = df['Cell Name'].astype('category').cat.codes
df['site_id']   = df['Radio Site'].astype('category').cat.codes
df['region_id'] = df['Region'].astype('category').cat.codes

# ── FDD/TDD one-hot encoding
df = pd.get_dummies(df, columns=['Cell FDD TDD Indication'])

# ── Lag constants
LAG_2W, LAG_1W, LAG_1D = 24*14, 24*7, 24

# ── 2-week lags (all three KPIs)
df['prb_lag_2w']        = df.groupby('Cell Name')['DL PRB Load'].shift(LAG_2W)
df['throughput_lag_2w'] = df.groupby('Cell Name')['LTE DL User Throughput'].shift(LAG_2W)
df['users_lag_2w']      = df.groupby('Cell Name')['Active Users'].shift(LAG_2W)

# ── 1-week lags
df['prb_lag_1w']        = df.groupby('Cell Name')['DL PRB Load'].shift(LAG_1W)
df['throughput_lag_1w'] = df.groupby('Cell Name')['LTE DL User Throughput'].shift(LAG_1W)
df['users_lag_1w']      = df.groupby('Cell Name')['Active Users'].shift(LAG_1W)

# ── 24-hour lags
df['prb_lag_1d']        = df.groupby('Cell Name')['DL PRB Load'].shift(LAG_1D)
df['throughput_lag_1d'] = df.groupby('Cell Name')['LTE DL User Throughput'].shift(LAG_1D)
df['users_lag_1d']      = df.groupby('Cell Name')['Active Users'].shift(LAG_1D)

# ── Rolling 24h mean for target KPI (shift 1 to avoid leakage)
df['throughput_rolling_24h'] = (
    df.groupby('Cell Name')['LTE DL User Throughput']
    .transform(lambda x: x.shift(1).rolling(24, min_periods=1).mean())
)

# ── Cyclical time encoding
df['hour_sin']   = np.sin(2*np.pi*df['Time'].dt.hour       / 24)
df['hour_cos']   = np.cos(2*np.pi*df['Time'].dt.hour       / 24)
df['dow_sin']    = np.sin(2*np.pi*df['Time'].dt.dayofweek  / 7)
df['dow_cos']    = np.cos(2*np.pi*df['Time'].dt.dayofweek  / 7)
df['month_sin']  = np.sin(2*np.pi*df['Time'].dt.month      / 12)
df['month_cos']  = np.cos(2*np.pi*df['Time'].dt.month      / 12)
df['is_weekend'] = (df['Time'].dt.dayofweek >= 5).astype(int)

# ── Cells with enough data in the test window (computed before ffill/bfill)
# Threshold: a full month (720h) if the test window is that long, otherwise
# 90% of however many unique timestamps actually exist in the test window.
_split_date    = df['Time'].quantile(0.8)
_test_window   = df[df['Time'] > _split_date]
_expected_hrs  = _test_window['Time'].nunique()
_min_hours     = min(720, int(0.9 * _expected_hrs))
_counts        = (
    _test_window.groupby('Cell Name')[TARGET_COL]
    .apply(lambda x: x.notna().sum())
)
full_month_cells = _counts[_counts >= _min_hours].index
if len(full_month_cells) == 0:
    full_month_cells = _counts.index   # fallback: use all cells
    print(f"Warning: no cells met ≥{_min_hours}h threshold; using all {len(full_month_cells)} cells.")
else:
    print(f"Test window: {_expected_hrs}h total  |  threshold: {_min_hours}h  |  qualifying cells: {len(full_month_cells)}")

df = df.ffill().bfill()

# ── Feature list (excludes current-hour value of the target KPI)
BASE_FEATURES = [
    # ── Cell / site / region identity
    'cell_id', 'site_id', 'region_id',
    # ── 24-hour lags (all three KPIs)
    'prb_lag_1d', 'throughput_lag_1d', 'users_lag_1d',
    # ── 1-week lags
    'prb_lag_1w', 'throughput_lag_1w', 'users_lag_1w',
    # ── 2-week lags
    'prb_lag_2w', 'throughput_lag_2w', 'users_lag_2w',
    # ── Rolling 24h mean for target KPI
    'throughput_rolling_24h',
    # ── Cyclical time features
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos', 'is_weekend',
] + [c for c in df.columns if 'Cell FDD TDD Indication_' in c]

print(f"Rows: {len(df):,}   Features: {len(BASE_FEATURES)}")

Test window: 106h total  |  threshold: 95h  |  qualifying cells: 25668
Rows: 13,712,735   Features: 22


## 3 · Evaluation Metrics (MAE / RMSE / MAPE)

In [13]:
def evaluate(y_true, y_pred):
    """Returns (MAE, RMSE, sMAPE).
    Uses Symmetric MAPE so near-zero target values don't inflate the error.
    sMAPE = mean(200 * |y-yhat| / (|y| + |yhat|)), bounded in [0, 200%]."""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.abs(y_true) + np.abs(y_pred)
    mask  = denom > 0
    smape = np.mean(200 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask])
    return round(mae, 4), round(rmse, 4), round(smape, 4)

## 4 · Train / Test Split (80 / 20 chronological)

In [14]:
split_date = df['Time'].quantile(0.8)
train_df   = df[df['Time'] <= split_date].copy()
test_df    = df[df['Time'] >  split_date].copy()
print(f"Train: {len(train_df):,}   Test: {len(test_df):,}   Split: {split_date}")

Train: 10,989,043   Test: 2,723,692   Split: 2026-04-09 00:00:00


## 5 · ARIMA
> ARIMA supports only univariate time series.  
> Trained & evaluated exclusively on the pilot cell for `LTE DL User Throughput`.

In [15]:
from pmdarima    import auto_arima
import statsmodels.api as sm

cell_df  = df[df['Cell Name'] == CELL_NAME].sort_values('Time').reset_index(drop=True)
n_split  = int(len(cell_df) * 0.8)
arima_tr = cell_df['LTE DL User Throughput'].iloc[:n_split]
arima_te = cell_df['LTE DL User Throughput'].iloc[n_split:]

print('Running auto_arima to find order …')
auto_mdl = auto_arima(arima_tr[:500], seasonal=False, stepwise=True,
                      suppress_warnings=True, error_action='ignore')
ORDER    = auto_mdl.order
ORDER    = (10,1,10)

print(f'Best ARIMA order: {ORDER}')

Running auto_arima to find order …
Best ARIMA order: (10, 1, 10)


In [16]:
def arima_walk_forward(train_series, test_series, horizon):
    history  = list(train_series)
    preds, actuals = [], []
    test_arr = test_series.values
    for i in range(0, len(test_arr) - horizon, horizon):
        mdl = sm.tsa.ARIMA(history, order=ORDER)
        fit = mdl.fit()
        fc  = fit.forecast(steps=horizon)
        preds.append(float(fc[-1]))
        actuals.append(float(test_arr[i + horizon - 1]))
        history.extend(test_arr[i:i+horizon].tolist())
    return np.array(actuals), np.array(preds)

print('ARIMA H6  walk-forward …')
act_h6,  pred_h6  = arima_walk_forward(arima_tr, arima_te, 6)
mae6, rmse6, mape6 = evaluate(act_h6, pred_h6)
print(f'  H6  → MAE={mae6}  RMSE={rmse6}  MAPE={mape6}%')

print('ARIMA H24 walk-forward …')
act_h24, pred_h24  = arima_walk_forward(arima_tr, arima_te, 24)
mae24, rmse24, mape24 = evaluate(act_h24, pred_h24)
print(f'  H24 → MAE={mae24}  RMSE={rmse24}  MAPE={mape24}%')

arima_metrics = {
    'H6 MAE':mae6,  'H6 RMSE':rmse6,  'H6 MAPE':mape6,
    'H24 MAE':mae24,'H24 RMSE':rmse24,'H24 MAPE':mape24
}

ARIMA H6  walk-forward …
  H6  → MAE=4882.2238  RMSE=5635.5936  MAPE=35.0954%
ARIMA H24 walk-forward …
  H24 → MAE=6268.8982  RMSE=7183.0255  MAPE=28.2375%


## 6 · Prepare Direct-Forecast Targets (H6 & H24)

In [17]:
results = {}

def make_horizon_dataset(horizon):
    """Shift target KPI forward by `horizon` steps per cell for direct forecasting.
    Training uses all cells; test is restricted to full_month_cells only."""
    d = df.copy()
    d['target'] = d.groupby('Cell Name')['LTE DL User Throughput'].shift(-horizon)
    d = d.dropna(subset=['target'])
    te = d[(d['Time'] > split_date) & (d['Cell Name'].isin(full_month_cells))]
    return d[d['Time'] <= split_date], te

print('Horizon datasets ready.')
print(f"Test evaluation restricted to {len(full_month_cells)} full-month cells.")

Horizon datasets ready.
Test evaluation restricted to 25668 full-month cells.


## 7 · ML Models — Linear Regression, Ridge, XGBoost, LightGBM
> **Random Forest excluded**: requires >1 GB contiguous RAM on 10M rows (MemoryError confirmed during initial testing).

In [18]:
import copy

ML_MODELS = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'XGBoost':  xgb.XGBRegressor(n_estimators=100, random_state=42,
                                  eval_metric='rmse', verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
}

ml_preds_h24  = {}    # model_name → H24 predictions on pilot cell test rows
xgb_model_h24 = None  # saved for feature importance
xgb_scaler    = None

for name, base_model in ML_MODELS.items():
    print(f"\n{'='*52}\n  {name}")
    m = {}
    for H in HORIZONS:
        tr_h, te_h = make_horizon_dataset(H)
        X_tr = tr_h[BASE_FEATURES].values;  y_tr = tr_h['target'].values
        X_te = te_h[BASE_FEATURES].values;  y_te = te_h['target'].values

        scaler  = StandardScaler()
        X_tr_s  = scaler.fit_transform(X_tr)
        X_te_s  = scaler.transform(X_te)

        model = copy.deepcopy(base_model)
        model.fit(X_tr_s, y_tr)
        preds = model.predict(X_te_s)

        mae, rmse, mape = evaluate(y_te, preds)
        m[f'H{H} MAE']  = mae
        m[f'H{H} RMSE'] = rmse
        m[f'H{H} MAPE'] = mape
        print(f"  H{H:2d} → MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

        if H == 24:
            zag_te  = te_h[te_h['Cell Name'] == CELL_NAME].sort_values('Time')
            zag_X   = scaler.transform(zag_te[BASE_FEATURES].values)
            ml_preds_h24[name] = model.predict(zag_X)
            if name == 'XGBoost':
                xgb_model_h24 = model
                xgb_scaler    = scaler

    results[name] = m

print("\nAll ML models done.")


  Linear Regression
  H 6 → MAE=5068.1862  RMSE=7433.0085  MAPE=34.55%
  H24 → MAE=4550.2160  RMSE=6981.6553  MAPE=28.73%

  Ridge Regression
  H 6 → MAE=5068.1867  RMSE=7433.0088  MAPE=34.55%
  H24 → MAE=4550.2158  RMSE=6981.6556  MAPE=28.73%

  XGBoost
  H 6 → MAE=4166.8389  RMSE=6512.4250  MAPE=25.92%
  H24 → MAE=4166.0259  RMSE=6604.7457  MAPE=24.49%

  LightGBM
  H 6 → MAE=4187.7338  RMSE=6541.0320  MAPE=26.10%
  H24 → MAE=4169.4012  RMSE=6617.9696  MAPE=24.60%

All ML models done.


## 8 · Neural Network

In [19]:
import tensorflow as tf
from keras.models    import Sequential
from keras.layers    import Dense, Dropout
from keras.callbacks import EarlyStopping
import keras.backend as K

nn_metrics   = {}
nn_preds_h24 = None

for H in HORIZONS:
    print(f"\nNeural Network H{H} …")
    K.clear_session()

    tr_h, te_h = make_horizon_dataset(H)
    X_tr = tr_h[BASE_FEATURES].values;  y_tr = tr_h['target'].values
    X_te = te_h[BASE_FEATURES].values;  y_te = te_h['target'].values

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_te_s  = scaler.transform(X_te)

    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_s.shape[1],)),
        Dropout(0.2),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(32,  activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(patience=3, restore_best_weights=True, verbose=0)
    model.fit(X_tr_s, y_tr, epochs=20, batch_size=512,
              validation_split=0.1, callbacks=[es], verbose=0)

    preds = model.predict(X_te_s, verbose=0).flatten()
    mae, rmse, mape = evaluate(y_te, preds)
    nn_metrics[f'H{H} MAE']  = mae
    nn_metrics[f'H{H} RMSE'] = rmse
    nn_metrics[f'H{H} MAPE'] = mape
    print(f"  H{H:2d} → MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    if H == 24:
        zag_te       = te_h[te_h['Cell Name'] == CELL_NAME].sort_values('Time')
        zag_X        = scaler.transform(zag_te[BASE_FEATURES].values)
        nn_preds_h24 = model.predict(zag_X, verbose=0).flatten()

results['Neural Network'] = nn_metrics
print("Neural Network done.")


Neural Network H6 …

  H 6 → MAE=4225.7659  RMSE=6554.9808  MAPE=26.84%

Neural Network H24 …
  H24 → MAE=4210.6364  RMSE=6672.7749  MAPE=25.21%
Neural Network done.


## 9 · Comparison Table

In [20]:
results['ARIMA'] = arima_metrics

ROW_ORDER = ['Linear Regression','Ridge Regression','ARIMA',
             'XGBoost','LightGBM','Neural Network']
COLS = ['H6 MAE','H6 RMSE','H6 MAPE','H24 MAE','H24 RMSE','H24 MAPE']

table = pd.DataFrame({m: results[m] for m in ROW_ORDER}).T[COLS]
table.index.name = 'Model'
print(table.to_string())
table.to_csv(TABLE_PATH)
print(f"\nSaved → {TABLE_PATH}")

                      H6 MAE    H6 RMSE  H6 MAPE    H24 MAE   H24 RMSE  H24 MAPE
Model                                                                           
Linear Regression  5068.1862  7433.0085  34.5491  4550.2160  6981.6553   28.7310
Ridge Regression   5068.1867  7433.0088  34.5491  4550.2158  6981.6556   28.7310
ARIMA              4882.2238  5635.5936  35.0954  6268.8982  7183.0255   28.2375
XGBoost            4166.8389  6512.4250  25.9185  4166.0259  6604.7457   24.4891
LightGBM           4187.7338  6541.0320  26.0987  4169.4012  6617.9696   24.5994
Neural Network     4225.7659  6554.9808  26.8392  4210.6364  6672.7749   25.2109

Saved → comparison_table_Throughput.csv


## 10 · Forecast Plots  (48h history + 24h prediction vs actual)

In [21]:
def forecast_plot(model_name, times_72, actual_72, pred_24, save_path):
    t_hist, t_fore = times_72.iloc[:48], times_72.iloc[48:]
    a_hist, a_fore = actual_72[:48],       actual_72[48:]

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(t_hist, a_hist, color='steelblue', linewidth=2,
            label='Actual (history 48h)')
    ax.plot(t_fore, a_fore, color='steelblue', linewidth=2, alpha=0.4,
            label='Actual (forecast window)')
    ax.plot(t_fore, pred_24, color='tomato', linewidth=2, linestyle='--',
            label='Predicted (24h ahead)')
    ax.axvspan(t_fore.iloc[0], t_fore.iloc[-1], alpha=0.07, color='orange')
    ax.axvline(x=t_fore.iloc[0], color='gray', linestyle=':', linewidth=1.5)
    ax.set_title(f"{model_name}  |  {CELL_NAME}", fontsize=13, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel(YLABEL)
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f'  Saved → {save_path}')

### 11a · ARIMA forecast plot

In [22]:
test_cell      = cell_df.iloc[n_split:].reset_index(drop=True)
window72       = test_cell.iloc[-72:].reset_index(drop=True)
boundary_idx   = n_split + len(test_cell) - 24
arima_fit_data = cell_df['LTE DL User Throughput'].iloc[:boundary_idx]
fit_plot       = sm.tsa.ARIMA(arima_fit_data, order=ORDER).fit()
arima_fc_24    = fit_plot.forecast(steps=24).values

forecast_plot(
    model_name = 'ARIMA',
    times_72   = window72['Time'],
    actual_72  = window72['LTE DL User Throughput'].values,
    pred_24    = arima_fc_24,
    save_path  = os.path.join(PLOT_DIR, 'ARIMA_LTE_DL_User_Throughput.png')
)

  Saved → throughput_plots\ARIMA_LTE_DL_User_Throughput.png


### 11b · ML & NN forecast plots

In [23]:
_, te_h24    = make_horizon_dataset(24)
zag_full     = te_h24[te_h24['Cell Name'] == CELL_NAME].sort_values('Time').reset_index(drop=True)
win72        = zag_full.iloc[-72:].reset_index(drop=True)
times_72_ml  = win72['Time']
actual_72_ml = win72['LTE DL User Throughput'].values
offset       = len(zag_full) - 72

for name in ['Linear Regression', 'Ridge Regression', 'XGBoost', 'LightGBM']:
    pred_24h  = ml_preds_h24[name][offset+24 : offset+48]
    safe_name = name.replace(' ', '_')
    forecast_plot(
        model_name = name,
        times_72   = times_72_ml,
        actual_72  = actual_72_ml,
        pred_24    = pred_24h,
        save_path  = os.path.join(PLOT_DIR, f'{safe_name}_LTE_DL_User_Throughput.png')
    )

if nn_preds_h24 is not None:
    forecast_plot(
        model_name = 'Neural Network',
        times_72   = times_72_ml,
        actual_72  = actual_72_ml,
        pred_24    = nn_preds_h24[offset+24 : offset+48],
        save_path  = os.path.join(PLOT_DIR, f'Neural_Network_LTE_DL_User_Throughput.png')
    )

print('All forecast plots saved to:', PLOT_DIR)

  Saved → throughput_plots\Linear_Regression_LTE_DL_User_Throughput.png
  Saved → throughput_plots\Ridge_Regression_LTE_DL_User_Throughput.png
  Saved → throughput_plots\XGBoost_LTE_DL_User_Throughput.png
  Saved → throughput_plots\LightGBM_LTE_DL_User_Throughput.png
  Saved → throughput_plots\Neural_Network_LTE_DL_User_Throughput.png
All forecast plots saved to: throughput_plots


In [24]:
## 11c · Combined Forecast — XGBoost vs LightGBM vs Neural Network (H+24)
fig, ax = plt.subplots(figsize=(16, 5))

t_hist = times_72_ml.iloc[:48]
t_fore = times_72_ml.iloc[48:]
a_hist = actual_72_ml[:48]
a_fore = actual_72_ml[48:]

xgb_fc  = ml_preds_h24['XGBoost'][offset+24  : offset+48]
lgbm_fc = ml_preds_h24['LightGBM'][offset+24 : offset+48]

ax.plot(t_hist, a_hist, color='steelblue', linewidth=2,
        label='Actual (history 48h)')
ax.plot(t_fore, a_fore, color='steelblue', linewidth=2, alpha=0.35,
        label='Actual (forecast window)')
ax.plot(t_fore, xgb_fc,  color='#E07B39', linewidth=2, linestyle='--',
        marker='o', markersize=4, label='XGBoost H+24')
ax.plot(t_fore, lgbm_fc, color='#3DA85A', linewidth=2, linestyle='--',
        marker='s', markersize=4, label='LightGBM H+24')
if nn_preds_h24 is not None:
    nn_fc = nn_preds_h24[offset+24 : offset+48]
    ax.plot(t_fore, nn_fc, color='#B03EB5', linewidth=2, linestyle='--',
            marker='^', markersize=4, label='Neural Network H+24')

ax.axvspan(t_fore.iloc[0], t_fore.iloc[-1], alpha=0.07, color='orange')
ax.axvline(x=t_fore.iloc[0], color='gray', linestyle=':', linewidth=1.5)
ax.set_title(f"XGBoost vs LightGBM vs Neural Network  |  {CELL_NAME}",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Time')
ax.set_ylabel(YLABEL)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
plt.tight_layout()

_save = os.path.join(PLOT_DIR, f'combined_forecast_{TARGET_COL.replace(" ", "_")}.png')
plt.savefig(_save, dpi=150)
plt.close()
print(f'Combined forecast plot saved → {_save}')

Combined forecast plot saved → throughput_plots\combined_forecast_LTE_DL_User_Throughput.png


## 12 · XGBoost Feature Importance (H+24)

In [25]:
if xgb_model_h24 is not None:
    importances = xgb_model_h24.feature_importances_
    # Sort descending
    idx         = np.argsort(importances)[::-1]
    sorted_feats = [BASE_FEATURES[i] for i in idx]
    sorted_vals  = importances[idx]

    # Color: top-30% features in dark blue, rest in light blue
    threshold = np.percentile(importances, 70)
    colors    = ['#185FA5' if v >= threshold else '#93B8D9' for v in sorted_vals]

    fig, ax = plt.subplots(figsize=(10, max(6, len(BASE_FEATURES)*0.35)))
    ax.barh(sorted_feats[::-1], sorted_vals[::-1], color=colors[::-1])
    ax.set_xlabel('Feature Importance (Gain)', fontsize=11)
    ax.set_title(f'XGBoost Feature Importance — LTE DL User Throughput (H+24)',
                fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=9)
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, 'fig_xgb_importance_Throughput.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Feature importance saved → {save_path}')
    print('\nTop-5 features:')
    for feat, val in zip(sorted_feats[:5], sorted_vals[:5]):
        print(f'  {feat:35s}  {val:.4f}')
else:
    print('XGBoost model not found — run section 7 first.')

Feature importance saved → throughput_plots\fig_xgb_importance_Throughput.png

Top-5 features:
  throughput_lag_1d                    0.7093
  throughput_rolling_24h               0.0976
  hour_sin                             0.0543
  throughput_lag_1w                    0.0282
  hour_cos                             0.0221


## 13 · Summary

In [26]:
print('\n' + '='*65)
print('  COMPARISON TABLE — LTE DL User Throughput')
print('='*65)
print(table.to_string())
print('='*65)
print(f'  Table → comparison_table_Throughput.csv')
print(f'  Plots → {PLOT_DIR}/')
for f in sorted(os.listdir(PLOT_DIR)):
    print(f'    {f}')


  COMPARISON TABLE — LTE DL User Throughput
                      H6 MAE    H6 RMSE  H6 MAPE    H24 MAE   H24 RMSE  H24 MAPE
Model                                                                           
Linear Regression  5068.1862  7433.0085  34.5491  4550.2160  6981.6553   28.7310
Ridge Regression   5068.1867  7433.0088  34.5491  4550.2158  6981.6556   28.7310
ARIMA              4882.2238  5635.5936  35.0954  6268.8982  7183.0255   28.2375
XGBoost            4166.8389  6512.4250  25.9185  4166.0259  6604.7457   24.4891
LightGBM           4187.7338  6541.0320  26.0987  4169.4012  6617.9696   24.5994
Neural Network     4225.7659  6554.9808  26.8392  4210.6364  6672.7749   25.2109
  Table → comparison_table_Throughput.csv
  Plots → throughput_plots/
    ARIMA_LTE_DL_User_Throughput.png
    LightGBM_LTE_DL_User_Throughput.png
    Linear_Regression_LTE_DL_User_Throughput.png
    Neural_Network_LTE_DL_User_Throughput.png
    Ridge_Regression_LTE_DL_User_Throughput.png
    XGBoost_LTE_D